In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

OUT_DIR = Path("out/audio/")
ART_DIR = Path("out/audio/audio_models")
ART_DIR.mkdir(parents=True, exist_ok=True)

OUT_ROOT = Path("preds_unimodal")
MODALITY = "audio"

LABELS = ["MF", "SK", "SJ"]
TAU = 0.2
USE_ONLY_MASK_ANY = False
PRESENT_SOURCE = "mask_any"

VAR_THRESH = 1e-6
BATCH = 512
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


In [ ]:
def load_windows(split: str) -> pd.DataFrame:
    path = OUT_DIR / f"windows_{split}_with_audio.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    df = pd.read_csv(path)

    if "w_center_x" in df.columns and "w_center_y" in df.columns:
        df.rename(columns={"w_center_x":"w_center"}, inplace=True)
        df.drop(columns=["w_center_y"], inplace=True)
    elif "w_center_x" in df.columns:
        df.rename(columns={"w_center_x":"w_center"}, inplace=True)
    elif "w_center_y" in df.columns:
        df.rename(columns={"w_center_y":"w_center"}, inplace=True)

    if USE_ONLY_MASK_ANY:
        if "mask_any" not in df.columns:
            raise RuntimeError("mask not present.")
        df = df[df["mask_any"] == 1].reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    return df

train = load_windows("train")
val   = load_windows("val")
test  = load_windows("test")

print("train/val/test:", len(train), len(val), len(test))
print("Columns example:", list(train.columns)[:15])


In [ ]:
DROP = {
    "video_id","w_start","w_end","w_center","split",
    "mask_any","is_bg_negative",
    "y_MF","y_SK","y_SJ","mask_MF","mask_SK","mask_SJ",
}

feat_cols = [c for c in train.columns if c not in DROP and pd.api.types.is_numeric_dtype(train[c])]
assert "has_any_audio_feature" in feat_cols, "Expected has_any_audio_feature in features"

print("Raw audio feature columns:", len(feat_cols))

mu = train[feat_cols].mean(skipna=True)
sd = train[feat_cols].std(skipna=True).replace(0, 1.0)

def transform(df):
    X = (df[feat_cols] - mu) / sd
    X = X.fillna(0.0)
    return X

X_train_full = transform(train)
X_val_full   = transform(val)
X_test_full  = transform(test)

var = X_train_full.var(axis=0)
keep = var[var > VAR_THRESH].index.tolist()

X_train = X_train_full[keep]
X_val   = X_val_full[keep]
X_test  = X_test_full[keep]

print("Kept features after variance filter:", len(keep))
print("X_train shape:", X_train.shape)

feature_meta = {
    "feat_cols_raw": feat_cols,
    "feat_cols_keep": keep,
    "var_thresh": VAR_THRESH,
}
with open(ART_DIR / "audio_feature_meta.pkl", "wb") as f:
    pickle.dump(feature_meta, f)
with open(ART_DIR / "audio_normalization.pkl", "wb") as f:
    pickle.dump({"mu": mu, "sd": sd}, f)

print("[OK] saved meta + normalization to:", ART_DIR)


In [ ]:
def make_targets_and_masks(df):
    Y = np.stack([(df[f"y_{lab}"].to_numpy() >= TAU).astype(np.float32) for lab in LABELS], axis=1)
    M = np.stack([df[f"mask_{lab}"].to_numpy().astype(np.float32) for lab in LABELS], axis=1)
    return Y, M

Y_train, M_train = make_targets_and_masks(train)
Y_val,   M_val   = make_targets_and_masks(val)
Y_test,  M_test  = make_targets_and_masks(test)

print("Y_train shape:", Y_train.shape, "M_train shape:", M_train.shape)
print("Train positive rates:", {lab: float(Y_train[:,i].mean()) for i,lab in enumerate(LABELS)})


In [ ]:
def masked_metrics(y_true, y_prob, mask, labels=("MF","SK","SJ")):
    rows = []
    for i, lab in enumerate(labels):
        m = mask[:, i] > 0.5
        yt = y_true[m, i]
        yp = y_prob[m, i]
        if len(np.unique(yt)) < 2:
            pr = np.nan
            roc = np.nan
        else:
            pr = average_precision_score(yt, yp)
            roc = roc_auc_score(yt, yp)
        rows.append({"label": lab, "PR_AUC": pr, "ROC_AUC": roc})
    return pd.DataFrame(rows)

def print_split_metrics(name, y_true, y_prob, mask):
    dfm = masked_metrics(y_true, y_prob, mask, labels=LABELS)
    print(f"\n{name}")
    display(dfm)
    return dfm

In [ ]:
def fit_logreg_per_label(X, Y, M, labels=("MF","SK","SJ"), seed=42):
    models = {}
    for i, lab in enumerate(labels):
        m = M[:, i] > 0.5
        Xi = X.loc[m].to_numpy()
        yi = Y[m, i].astype(int)

        if len(np.unique(yi)) < 2:
            models[lab] = None
            continue

        classes = np.array([0, 1])
        w = compute_class_weight(class_weight="balanced", classes=classes, y=yi)
        class_weight = {0: w[0], 1: w[1]}

        clf = LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            max_iter=2000,
            class_weight=class_weight,
            random_state=seed,
        )
        clf.fit(Xi, yi)
        models[lab] = clf

    return models

def predict_logreg_probs(models, X, labels=("MF","SK","SJ")):
    P = np.zeros((len(X), len(labels)), dtype=np.float32)
    for i, lab in enumerate(labels):
        clf = models.get(lab)
        if clf is None:
            P[:, i] = np.nan
        else:
            P[:, i] = clf.predict_proba(X.to_numpy())[:, 1].astype(np.float32)
    return P

logreg_models = fit_logreg_per_label(X_train, Y_train, M_train, labels=LABELS, seed=SEED)

P_val_lr  = predict_logreg_probs(logreg_models, X_val, labels=LABELS)
P_test_lr = predict_logreg_probs(logreg_models, X_test, labels=LABELS)

print_split_metrics("LogReg — VAL",  Y_val,  P_val_lr,  M_val)
print_split_metrics("LogReg — TEST", Y_test, P_test_lr, M_test)

with open(ART_DIR / "audio_logreg_models.pkl", "wb") as f:
    pickle.dump(logreg_models, f)


In [ ]:
class AudioDataset(Dataset):
    def __init__(self, X: pd.DataFrame, Y: np.ndarray, M: np.ndarray):
        self.X = torch.tensor(X.to_numpy(), dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
        self.M = torch.tensor(M, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.M[idx]

ds_train = AudioDataset(X_train, Y_train, M_train)
ds_val   = AudioDataset(X_val,   Y_val,   M_val)
ds_test  = AudioDataset(X_test,  Y_test,  M_test)

train_loader = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False, num_workers=0)

pos_counts = []
neg_counts = []
for i in range(3):
    m = (M_train[:, i] > 0.5)
    y = Y_train[m, i]
    pos = float(y.sum())
    neg = float(len(y) - y.sum())
    pos_counts.append(pos)
    neg_counts.append(neg)

pos_weight = torch.tensor([(neg_counts[i] / max(pos_counts[i], 1e-6)) for i in range(3)], dtype=torch.float32, device=DEVICE)
print("pos_weight:", pos_weight)


In [ ]:
class MLP(nn.Module):
    def __init__(self, d_in: int, d_hidden: int = 256, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, 3),
        )

    def forward(self, x):
        return self.net(x)

def masked_bce_with_logits(logits, y, m, pos_weight):

    loss_raw = nn.functional.binary_cross_entropy_with_logits(
        logits, y, reduction="none", pos_weight=pos_weight
    )

    loss_masked = loss_raw * m
    denom = m.sum().clamp(min=1.0)
    return loss_masked.sum() / denom

@torch.no_grad()
def predict_logits_probs_from_loader(model, loader, device):
    model.eval()
    all_logits, all_probs, all_y, all_m = [], [], [], []
    for x, y, m in loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits)
        all_logits.append(logits.detach().cpu().numpy())
        all_probs.append(probs.detach().cpu().numpy())
        all_y.append(y.detach().cpu().numpy())
        all_m.append(m.detach().cpu().numpy())
    L = np.concatenate(all_logits, axis=0).astype(np.float32)
    P = np.concatenate(all_probs,  axis=0).astype(np.float32)
    Y = np.concatenate(all_y,      axis=0).astype(np.float32)
    M = np.concatenate(all_m,      axis=0).astype(np.float32)
    return L, P, Y, M


In [ ]:
model = MLP(d_in=X_train.shape[1], d_hidden=256, dropout=0.2).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best = {"epoch": -1, "state": None, "val_mean_pr": -np.inf}
bad = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_den = 0.0

    for x, y, m in train_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        m = m.to(DEVICE)

        opt.zero_grad()
        logits = model(x)
        loss = masked_bce_with_logits(logits, y, m, pos_weight)
        loss.backward()
        opt.step()

        total_loss += float(loss.item()) * float(m.sum().item())
        total_den  += float(m.sum().item())

    train_loss = total_loss / max(total_den, 1.0)

    Lva, Pva, Yva, Mva = predict_logits_probs_from_loader(model, val_loader, DEVICE)
    val_df = masked_metrics(Yva, Pva, Mva, labels=LABELS)
    val_mean_pr = float(np.nanmean(val_df["PR_AUC"].to_numpy()))

    print(f"epoch {epoch:02d} | loss {train_loss:.4f} | val mean PR-AUC {val_mean_pr:.4f}")

    if val_mean_pr > best["val_mean_pr"] + 1e-6:
        best.update({"epoch": epoch, "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, "val_mean_pr": val_mean_pr})
        bad = 0
    else:
        bad += 1
        if bad >= PATIENCE:
            break



In [ ]:
model.load_state_dict({k: v.to(DEVICE) for k, v in best["state"].items()})
model.eval()

Lte, Pte, Yte, Mte = predict_logits_probs_from_loader(model, test_loader, DEVICE)
print_split_metrics("MLP — TEST", Yte, Pte, Mte)

torch.save({"state_dict": best["state"], "config": {"d_in": X_train.shape[1]}}, ART_DIR / "audio_mlp_model.pt")
with open(ART_DIR / "audio_mlp_best.pkl", "wb") as f:
    pickle.dump(best, f)

In [ ]:
def export_unimodal_preds(
    out_root: Path,
    split: str,
    modality: str,
    logits: np.ndarray,
    probs: np.ndarray,
    present: np.ndarray,
    windows_df: pd.DataFrame,
):
    out = Path(out_root) / split / modality
    out.mkdir(parents=True, exist_ok=True)

    keys = windows_df[["video_id","w_start","w_end","w_center"]].copy()
    keys.to_csv(out / f"keys_{split}.csv", index=False)

    logits = np.asarray(logits, dtype=np.float32)
    probs  = np.asarray(probs, dtype=np.float32)
    present = np.asarray(present, dtype=np.float32)

    assert logits.shape == (len(windows_df), 3), (logits.shape, len(windows_df))
    assert probs.shape  == (len(windows_df), 3), (probs.shape, len(windows_df))
    assert present.shape == (len(windows_df),), (present.shape, len(windows_df))

    np.save(out / f"logits_{split}.npy", logits)
    np.save(out / f"probs_{split}.npy", probs)

    conf = np.nanmax(probs, axis=1).astype(np.float32)
    np.save(out / f"conf_{split}.npy", conf)

    np.save(out / f"present_{split}.npy", present)

    print(f"[OK] Saved unimodal preds -> {out}")
    print("Shapes:", logits.shape, probs.shape, present.shape)

def get_present_mask(df: pd.DataFrame) -> np.ndarray:
    if PRESENT_SOURCE not in df.columns:
        raise RuntimeError(f"PRESENT_SOURCE='{PRESENT_SOURCE}' not in df columns.")
    return (df[PRESENT_SOURCE].to_numpy() > 0.5).astype(np.float32)


In [ ]:
export_train_loader = DataLoader(ds_train, batch_size=BATCH, shuffle=False, num_workers=0)
export_val_loader   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=0)
export_test_loader  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False, num_workers=0)

model.load_state_dict({k: v.to(DEVICE) for k, v in best["state"].items()})
model.eval()

Ltr, Ptr, Ytr, Mtr = predict_logits_probs_from_loader(model, export_train_loader, DEVICE)
Lva, Pva, Yva, Mva = predict_logits_probs_from_loader(model, export_val_loader, DEVICE)
Lte, Pte, Yte, Mte = predict_logits_probs_from_loader(model, export_test_loader, DEVICE)

present_tr = get_present_mask(train)
present_va = get_present_mask(val)
present_te = get_present_mask(test)

export_unimodal_preds(OUT_ROOT, "train", MODALITY, Ltr, Ptr, present_tr, train)
export_unimodal_preds(OUT_ROOT, "val",   MODALITY, Lva, Pva, present_va, val)
export_unimodal_preds(OUT_ROOT, "test",  MODALITY, Lte, Pte, present_te, test)

print("Shapes:")
print(" train logits", Ltr.shape, "probs", Ptr.shape, "present", present_tr.shape)
print(" val   logits", Lva.shape, "probs", Pva.shape, "present", present_va.shape)
print(" test  logits", Lte.shape, "probs", Pte.shape, "present", present_te.shape)


In [ ]:
def logreg_logits_probs(models, X: pd.DataFrame):
    N = len(X)
    logits = np.zeros((N, 3), dtype=np.float32)
    probs  = np.zeros((N, 3), dtype=np.float32)
    for i, lab in enumerate(LABELS):
        clf = models.get(lab)
        if clf is None:
            logits[:, i] = np.nan
            probs[:, i] = np.nan
        else:
            logits[:, i] = clf.decision_function(X.to_numpy()).astype(np.float32)
            probs[:, i]  = clf.predict_proba(X.to_numpy())[:, 1].astype(np.float32)
    return logits, probs

Ltr_lr, Ptr_lr = logreg_logits_probs(logreg_models, X_train)
Lva_lr, Pva_lr = logreg_logits_probs(logreg_models, X_val)
Lte_lr, Pte_lr = logreg_logits_probs(logreg_models, X_test)

export_unimodal_preds(OUT_ROOT, "train", "audio_logreg", Ltr_lr, Ptr_lr, present_tr, train)
export_unimodal_preds(OUT_ROOT, "val",   "audio_logreg", Lva_lr, Pva_lr, present_va, val)
export_unimodal_preds(OUT_ROOT, "test",  "audio_logreg", Lte_lr, Pte_lr, present_te, test)
